In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import random

# some calculations are sloooow. set this to True to reduce samples width to 
# increase speed, when you only care about the later calculation
speedup = True
apply_noise = False

# the paths of the images to be analyzed, comment
# some outto make the computation and debugging faster
paths = [
    #"01.png",
    #"02.png",
    "03.png",
    #"04.png",
    #"05.png",
    #"06.png",
    #"07.png",
]

ip = 0 # the path index for the single image graphs

# load images and show
print()
print(f"(A)")

S = [mpimg.imread(path) for path in paths]
H, W = S[0].shape
N = H * W

for i in range(len(S)):
    s = S[i]
    
    if s.ndim == 3:
        s = s.mean(axis=2)
    
    if apply_noise:
        noise_max = (s.min() + s.max())
        for y in range(H):
            for x in range(W):
                noise = np.random.uniform(0, noise_max);
                s[y, x] = s[y, x] + noise
    
    S[i] = s
    
plt.imshow(S[ip], cmap="gray")
plt.title("$S(x, y)$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"min: {S[ip].min()} max: {S[ip].max()} mean: {S[ip].mean()}")

# zero mean
print()
print(f"(B)")

S = (S - np.mean(S)) / np.std(S)

print(f"min: {S[ip].min()} max: {S[ip].max()} mean: {S[ip].mean()}")

# correlation between neighboring pixels
print()
print(f"(C)")

steps = 1
if speedup:
    steps = 32

d_values = np.arange(0, W-1, steps)
y_values = np.zeros(len(d_values))

for sindex, s in enumerate(S):
    for i, d in enumerate(d_values):
        print(f"progress... {sindex + 1}/{len(S)} {i + 1}/{len(d_values)}")
        
        temp = np.zeros((H, W - d))
        H_, W_ = temp.shape
        N_ = H_ * W_
        
        for y in range(H):
            for x in range(W - d):
                temp[y, x] = s[y, x] * s[y, x + d] / N_
        
        RS = temp.sum()
        y_values[i] += RS

y_values /= len(S)

plt.plot(d_values, y_values)
plt.title("correlation of neighboring pixels")
plt.xlabel("distance $|x_1 - x_2|$ (pixels)")
plt.ylabel("$R^S(x_1-x_2)$")
plt.show()

# furiour power specturm 1-D
print()
print(f"(D)")

steps = 1
if speedup:
    steps = 16

L = np.arange(0, int(W/2), steps)
#k_values = L * 2 * np.pi / W

x_values = []
y_values = []

for sindex, s in enumerate(S):

    #for i, k in enumerate(k_values):
    for i, l in enumerate(L):
        k = l * 2 * np.pi / W
        print(f"progress... {sindex + 1}/{len(S)} {i + 1}/{len(L)}")
        
        sum_c = 0
        sum_s = 0
        
        for y in range(H):
            for x in range(W):
                val_c = s[y, x] * np.cos(k * x)
                val_s = s[y, x] * np.sin(k * x)
                
                sum_c += val_c
                sum_s += val_s
        
        s2 = sum_c * sum_c + sum_s * sum_s
        
        x_values.append(l)
        y_values.append(s2)

print("scatter...")

plt.scatter(x_values, y_values, s=1)
plt.title("Fourier power spectrum (1D)")
plt.xlabel("$|k|$ (cycles/image)")
plt.ylabel("signal power $|S(k)|^2$")
plt.xscale('log')
plt.yscale('log')
plt.show()

# furiour power specturm 2-D
print()
print(f"(E)")

plt.imshow(S[ip], cmap="gray")
plt.title("original image $S$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

SFurier = np.fft.fft2(S[ip])

kx = np.fft.fftshift(np.fft.fftfreq(W)) * W
ky = np.fft.fftshift(np.fft.fftfreq(H)) * H
im = plt.imshow(
    np.log(np.abs(np.fft.fftshift(SFurier))),
    extent=[kx[0], kx[-1], ky[0], ky[-1]],
)
plt.title("Fourier components $\log |\mathcal{S}(k)|$")
plt.xlabel("$k_x$ (cycles/image)")
plt.ylabel("$k_y$ (cycles/image)")
plt.colorbar(im)
plt.show()

S_ = np.real(np.fft.ifft2(SFurier))

plt.imshow(S_, cmap="gray")
plt.title("$S'$ after the inverse Fourier transform of the Fourier components")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"sanity check: the difference should be 0 (due to rounding errors, close to 0")
Sdiff = S[ip] - S_
im = plt.imshow(Sdiff)
plt.title("$S - S'$")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(im)
plt.show()

# noise
print()
print(f"(F)")
print(f"noise control is at the top of this file")

#####################
print()
print("reached end!")